In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Setup

In second-order-odes.ipynb we saw that Newton's second law is a second-order ODE, which we can integrate by treating the position and velocity together as one state $y = (x, v)$.

In [ ]:
class FirstOrderODE:
    '''
    A first-order ODE specifies the rate of change of a state y(t)
    in terms of
     - the state itself, y
     - the time, t.

    The generic form for an ODE is

    dy
    --  = f(t, y)
    dt

    '''

    def f(self, t, y):
        raise NotImplementedError('The generic ODE has an unspecified slope function f.')

    def exact_solution(self, t, *args):
        raise NotImplementedError('The exact solution is not known (or at least not provided).')

In [ ]:
class SecondOrderODE(FirstOrderODE):
    '''
    A second-order ODE specifies the acceleration

    d^2x
    ----  = a(t, x, v)
    dt^2

    where v = dx/dt is the velocity.  Treating the state as y = (x, v) turns it into
    a first-order ODE,

    d  ( x )   (    v     )
    -- (   ) = (          )
    dt ( v )   ( a(t,x,v) )

    so a subclass only needs to provide the acceleration.
    '''

    def acceleration(self, t, x, v):
        raise NotImplementedError('The generic second-order ODE has an unspecified acceleration.')

    def f(self, t, y):
        # Unpack the state into its two components,
        x, v = y
        # and pack their rates of change back up the same way.
        return np.array([v, self.acceleration(t, x, v)])

In potential-energy.ipynb we saw that when the acceleration depends only on the position it comes from a potential energy $V(x)$,
\begin{align}
    a &= -V'(x)
    &
    E &= \frac{1}{2} v^2 + V(x)
\end{align}
and the energy $E$ is conserved.  As always, everything is per unit mass.

In [ ]:
class ConservativeSystem(SecondOrderODE):
    '''
    A conservative system has a potential energy V(x), and its acceleration is

    a = -V'(x)

    which doesn't depend on the velocity or the time.  Its energy

    E = 1/2 v^2 + V(x)

    is conserved.  A subclass provides the potential and the acceleration (minus its slope!).
    Everything is per unit mass.
    '''

    def potential(self, x):
        raise NotImplementedError('The generic conservative system has an unspecified potential.')

    def energy(self, x, v):
        return 0.5 * v**2 + self.potential(x)

We also wrote a check that a system's `acceleration` really is minus the slope of its `potential`.

In [ ]:
def slope(function, x, Delta_x=1e-5):
    '''
    Estimates the derivative of a function at x with a central difference.
    '''
    return (function(x + Delta_x) - function(x - Delta_x)) / (2 * Delta_x)

def acceleration_matches_potential(system, x):
    '''
    Checks whether a conservative system's acceleration is minus the slope of its potential at every x.
    '''
    return np.allclose(system.acceleration(0, x, 0), -slope(system.potential, x))

Finally, we need integrators.  In symplectic-integrators.ipynb we saw that RK2 lets the energy drift, while velocity Verlet does not.

In [ ]:
class FirstOrderODEIntegrator:
    '''
    An integrator is an algorithm for producing numerical solutions to ODEs.
    '''

    def __init__(self, ODE):

        self.ODE = ODE

    def starting_from(self, t_0, y_0):
        raise NotImplementedError('No integration algorithm has been specified.')

In [ ]:
class RungeKutta2(FirstOrderODEIntegrator):

    def __init__(self, ODE, alpha, Delta_t, steps):

        # Conditions for matching the Taylor expansion to 2nd order
        self.alpha = alpha
        self.b = 1/(2*self.alpha)
        self.a = 1 - self.b

        self.Delta_t = Delta_t
        self.steps = steps

        super().__init__(ODE)

    def starting_from(self, t_0, y_0):
        # Populate all the t values
        t = t_0 + self.Delta_t * np.arange(self.steps+1)
        # and allocate an array to store the y states we compute.
        y = np.zeros((self.steps+1,) + np.shape(y_0))

        # The user picked a starting y value
        y[0] = y_0

        # and we can step through the RK2 procedure over and over again.
        for step in range(self.steps):
            k_1 = self.ODE.f(t[step], y[step])
            k_2 = self.ODE.f(t[step] + self.alpha * self.Delta_t, y[step] + self.alpha * k_1 * self.Delta_t)

            y[step+1] = y[step] + (self.a * k_1 + self.b * k_2) * self.Delta_t

        return t, y

In [ ]:
class VelocityVerlet(FirstOrderODEIntegrator):
    '''
    Velocity Verlet splits each step into a half kick, a drift, and another half kick,

    v_{n+1/2} = v_n       + 1/2 a(t_n, x_n) Δt
    x_{n+1}   = x_n       + v_{n+1/2} Δt
    v_{n+1}   = v_{n+1/2} + 1/2 a(t_{n+1}, x_{n+1}) Δt

    It needs the ODE's acceleration, so it only works on second-order ODEs.
    '''

    def __init__(self, ODE, Delta_t, steps):

        self.Delta_t = Delta_t
        self.steps = steps

        super().__init__(ODE)

    def starting_from(self, t_0, y_0):

        # Populate all the t values
        t = t_0 + self.Delta_t * np.arange(self.steps+1)
        # and allocate an array with room for a whole state at each time.
        y = np.zeros((self.steps+1,) + np.shape(y_0))

        # The user picked a starting state
        y[0] = y_0

        for step in range(self.steps):
            x, v = y[step]
            # (acceleration also wants a velocity; we hand it the newest one we have.
            #  For forces that depend only on position it doesn't matter.)
            v_half = v      + 0.5 * self.ODE.acceleration(t[step],   x,     v)      * self.Delta_t   # half kick
            x_new  = x      +       v_half                                           * self.Delta_t   # drift
            v_new  = v_half + 0.5 * self.ODE.acceleration(t[step+1], x_new, v_half) * self.Delta_t   # half kick
            y[step+1] = np.array([x_new, v_new])

        return t, y

# The Morse Potential

Two atoms bound together into a molecule, like the two hydrogen atoms in H$_2$, sit a comfortable distance apart called the *bond length*.  Squeeze them closer together and they repel one another fiercely.  Pull them apart and they attract.  But that attraction doesn't last forever.  Pull them far enough apart and they stop noticing each other at all: the bond is broken, and the molecule has *dissociated*.

A spring can't describe that.  A spring pulls back harder and harder the further you stretch it and it never lets go.  In 1929 Philip Morse proposed a simple potential that can,
\begin{align}
    V(x) &= D \left(1 - e^{-\alpha x}\right)^2.
\end{align}
Here $x$ is how far the bond is *stretched* from its comfortable length, so that $x=0$ is equilibrium, $x<0$ is squeezed, and $x>0$ is stretched.  The two parameters are
 - $D$, the *dissociation energy*: when $x$ is very large $V \to D$.  Since $V(0) = 0$ the dissociation energy $D$ is how much energy it costs to pull the atoms apart starting from rest at equilibrium;
 - $\alpha$, which controls how narrow the well is.

Implement the Morse potential and its acceleration.

In [ ]:
class Morse(ConservativeSystem):
    '''
    The Morse potential V = D (1 - exp(-α x))^2 models the bond in a diatomic molecule,
    where x is how far the bond is stretched from its equilibrium length.
    '''

    def __init__(self, D, alpha):
        self.D = D
        self.alpha = alpha

    def potential(self, x):
        raise NotImplementedError("This function should compute the Morse potential V(x).")

    def acceleration(self, t, x, v):
        raise NotImplementedError("This function should compute the acceleration a = -V'(x).")

Before going any further, check your calculus.  This should print `True`.

In [ ]:
morse = Morse(0.5, 1.0)

x = np.linspace(-0.5, 6, 27)
print(acceleration_matches_potential(morse, x))

Near the bottom of any well a potential looks like a parabola, and so small vibrations are harmonic.  Taylor expand the Morse potential around $x=0$ and compare with the harmonic oscillator's potential $\frac{1}{2}\omega_0^2 x^2$.

**What is the angular frequency $\omega_0$ of small vibrations, in terms of $D$ and $\alpha$?**

In [ ]:
omega_0 = ... # the angular frequency of small vibrations, in terms of morse.D and morse.alpha
print(omega_0)

We picked $D = 0.5$ and $\alpha = 1$ so that this comes out to a nice number.

# Reading the Potential

We'll study molecules with these energies, given as fractions of the dissociation energy.  The last few are barely holding together.

In [ ]:
E = morse.D * np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 0.999])

A bound molecule vibrates between two turning points, where $V(x) = E$.  Solve that equation for $x$.  There are two solutions when $E < D$: an *inner* turning point with $x<0$, where the atoms are squeezed together, and an *outer* turning point with $x>0$, where the bond is stretched.

In [ ]:
def turning_points(morse, E):
    '''
    Returns the tuple (inner, outer) of turning points of a Morse molecule with energy E < D.
    '''
    raise NotImplementedError("This function should compute the two turning points, where V(x) = E.")
    return (inner, outer)

inner, outer = turning_points(morse, E)
print(inner)
print(outer)

Let's plot the Morse potential for $-1 \leq x \leq 8$ and draw each molecule's energy as a horizontal line that reaches from its inner turning point to its outer turning point.  We can also draw the dissociation energy $D$.

In [ ]:
x = np.linspace(-1, 8, 500)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(x, morse.potential(x), color='black')
ax.axhline(morse.D, color='gray', linestyle='dashed', label='D')

for i in range(len(E)):
    ax.plot([inner[i], outer[i]], [E[i], E[i]], color=f'C{i}', label=f'E = {E[i] / morse.D:.3f} D')

ax.set_ylim(0, 1.5 * morse.D)
ax.set_xlabel('x')
ax.set_ylabel('V(x)')
ax.legend(loc='lower right')

**Describe what you see.  How do the two turning points move as the energy goes up?  What happens to each of them if $E > D$, and what motion does that describe?**

# Holding a Molecule Together

Now let's integrate.  Start every molecule at rest at its inner turning point, so that it has exactly the energy we want.  Integrate all of them at once to $t = 1000$ with $\Delta t = 0.1$, once with RK2 (use $\alpha_\text{RK2} = 0.5$) and once with velocity Verlet.

In [ ]:
Delta_t = 0.1
steps = 10000

y_0 = ... # every molecule at rest at its inner turning point

t, rk2_y    = ... # the RK2 solution
t, verlet_y = ... # the velocity Verlet solution

print(rk2_y.shape)
print(verlet_y.shape)

For each integrator plot the stretch $x$ of every molecule as a function of time, and the energy of every molecule as a fraction of the dissociation energy, $E(t)/D$, as a function of time.

In [ ]:
raise NotImplementedError('''
For each integrator plot x(t) and E(t)/D for every molecule.
''')

**Describe what you see.  Which molecules does each integrator hold together?  For any molecule that falls apart, explain what went wrong using the potential plot above.  Is the breakup physical?  Would a smaller $\Delta t$ fix it?**

# How Fast Does a Molecule Vibrate?

A harmonic oscillator has the same frequency no matter its energy.  The Morse molecule doesn't: its angular frequency is known exactly,
\begin{align}
    \omega(E) &= \omega_0 \sqrt{1 - E/D}.
\end{align}
Let's test this formula.  In second-order-odes.ipynb we measured the period of a batch of pendulums all at once, by finding the first step where each one's velocity changed sign.  Use the same idea to measure the angular frequency of Morse molecules with the thirty energies below.

The molecules start at rest at the inner turning point, so first they move outward ($v > 0$) and half a period later they reach the outer turning point and come to rest again.

Use velocity Verlet.  You'll want a smaller step than before, $\Delta t = 0.01$, because you can only measure a time to within one step.  Make sure you integrate long enough for the slowest molecule to complete half a period!

In [ ]:
E_many = morse.D * np.linspace(0.05, 0.99, 30)

omega_measured = ... # the angular frequency of each molecule, measured from a velocity Verlet solution

print(omega_measured)

Plot your measured frequencies and the exact formula as a function of $E/D$.

In [ ]:
raise NotImplementedError('Plot the measured and exact angular frequencies as a function of E/D.')

**Describe what you see.  Why does the frequency match $\omega_0$ at low energy?  Why does it drop as the energy goes up, and what does it mean that $\omega \to 0$ as $E \to D$?**

# Hot Molecules Are Bigger

Most things expand when you heat them up.  Heating a molecule gives it more energy to vibrate with.  So, does a molecule with more energy have a longer bond, on average?

To answer that we need to say what "on average" means for something that never sits still.  Imagine checking on the molecule at lots of evenly spaced moments, writing down its stretch each time, and averaging everything you wrote down.  If we check more and more often until we monitor continuously, the average we want is the limit, or *the integral*
\begin{align}
    \bar{x} &= \frac{1}{T} \int_0^T x(t)\, dt \approx \frac{1}{T} \sum_{n=0}^{N-1} x_n \; \Delta t_n ,
    &
    T &= \sum_{n=0}^{N-1} \Delta t_n
\end{align}
where the $x_n$ are the stretches and $\Delta t_n$ their corresponding time steps.  A place where the molecule lingers counts for a lot.  A place it rushes through hardly counts at all.

Our integrators record $x$ at evenly spaced times.  So all of our $\Delta t_n$ are the same,
\begin{align}
    \bar{x} &\approx \frac{1}{T} \sum_{n=0}^{N-1} x_n \; \Delta t
    = \frac{\Delta t}{T} \sum_{n=0}^{N-1} x_n
    = \frac{1}{N} \sum_{n=0}^{N-1} x_n
    = \texttt{np.mean(}x\texttt{)}
\end{align}
for each trajectory.
So the time average is just the mean of the recorded values!
Since time is the first axis, we really need something like `np.mean(x, axis=0)`, which will give us the average for every trajectory.

Integrate the twelve molecules from before, each starting at rest at its inner turning point, with velocity Verlet to $t = 1000$.  That covers several vibrations of even the slowest molecule, so it doesn't matter much that the last vibration is cut off partway through.  Compute the time average of the stretch $x$ for each molecule and plot it as a function of $E/D$.

But how do we know our averages are right?  We don't have an exact answer to compare against.  What we can do is take a *continuum limit*: repeat the whole calculation with smaller and smaller steps, and draw every run on the same plot.  Once the answers stop changing appreciably as $\Delta t$ shrinks, we can trust them.

In [ ]:
Delta_ts = [0.1, 0.05, 0.02, 0.01, 0.005]

raise NotImplementedError('For each Δt, plot the time-averaged stretch of each molecule as a function of E/D.')

**Which molecules need a small $\Delta t$ before their average stops changing, and why?  Which $\Delta t$ would you trust?**

**Describe what you see.  Explain it using the shape of the potential.  What would this plot look like for a harmonic oscillator?**